# Coaching Agent — Demo Notebook

**Stack:** gemma3:4b (Ollama) · ChromaDB · HuggingFace Embeddings · LangChain

**Pipeline:**
```
PatientContext → RAG Retrieval → gemma3:4b → Polish → CoachingOutput
```

---
### ⚠️ Before running: make sure Ollama is running
```bash
ollama serve
ollama pull gemma3:4b
```

## 1. Setup

In [6]:
import sys
sys.path.insert(0, '.')  # ensure local modules are importable

from coaching_agent import CoachingAgent
from rag_retriever import CoachingKnowledgeBase
from schemas import PatientContext, RehabPhase, ConditionCategory, ExerciseRecord, make_sample_context
from upstream_adapter import merge_to_patient_context, validate_inputs

print('All modules loaded ✓')

All modules loaded ✓


## 2. Build / Load Knowledge Base

First run: indexes `dataset/clean` → saves to `./chroma_coaching_db`

Subsequent runs: loads from disk (fast)

In [2]:
kb = CoachingKnowledgeBase(
    data_dir='dataset/clean',        # ← your data directory
    persist_dir='./chroma_coaching_db',
    chunk_size=500,
    chunk_overlap=50,
).load_or_build(
    force_rebuild=False              # set True to re-index from scratch
)

Loading embeddings: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading existing vector DB from: ./chroma_coaching_db
Loaded 33782 document chunks


## 3. Initialise Agent

In [3]:
agent = CoachingAgent(
    knowledge_base=kb,
    ollama_model='gemma3:4b',
    retrieval_k=3,        # Keep at 3 to avoid memory pressure with 4b model
    enable_polish=True,   # Set False to skip 2nd LLM call (faster, less memory)
    verbose=True,
)

CoachingAgent ready — model: gemma3:4b, retrieval_k: 3


## 4. Run — Sample Scenarios

### Scenario A: Knee Osteoarthritis (Week 10)

In [ ]:
# --- Scenario A: Early Phase — Week 3, first time doing squats
#     Problem: can't control knee caving (valgus collapse), pain is high
coaching_event_early = {
    "coaching_event": {
        "event_id": "session_early_event_1",
        "timestamp": 8.5,
        "frame_index": 102,
        "exercise": {"name": "mini squat", "confidence": 0.91},
        "mistake": {
            "type": "knee valgus",
            "confidence": 0.82,
            "duration_seconds": 6.3,
            "persistence_rate": 0.74,
            "occurrences": 31,
        },
        "metrics": {
            "speed_rps": 0.6,
            "rom_level": 1,
            "height_level": 2,
            "torso_rotation": 0,
            "direction": "none",
            "no_obvious_issue_p": 0.05,
        },
        "quality_score": 0.28,
        "severity": "high",
        "is_recoaching": False,
        "session_time_minutes": 0.14,
        "tier": "tier_3",
        "cache_key": None,
        "routing_reason": "high severity mistake needs RAG context",
    },
    "session_id": "session_P001_early",
    "coaching_history": [],
}

patient_profile_early = {
    "condition": "knee osteoarthritis",
    "condition_category": "knee",
    "rehab_phase": "early",
    "pain_level": 7, # use david's pain scale pic
    "weeks_into_rehab": 3,
    "age": 58,
    "goals": "Walk dog 30 minutes daily without pain",
}

# --- Scenario B: Mid Phase — Week 10, progressed to full squat
#     Problem: forward lean (same squat pattern, less severe)
coaching_event_mid = {
    "coaching_event": {
        "event_id": "session_mid_event_1",
        "timestamp": 10.33,
        "frame_index": 155,
        "exercise": {"name": "squat", "confidence": 0.85},
        "mistake": {
            "type": "forward lean",
            "confidence": 0.38,
            "duration_seconds": 4.1,
            "persistence_rate": 0.31,
            "occurrences": 47,
        },
        "metrics": {
            "speed_rps": 1.0,
            "rom_level": 2,
            "height_level": 3,
            "torso_rotation": 1,
            "direction": "none",
            "no_obvious_issue_p": 0.1,
        },
        "quality_score": 0.35,
        "severity": "medium",
        "is_recoaching": False,
        "session_time_minutes": 0.17,
        "tier": "tier_2",
        "cache_key": None,
        "routing_reason": "medium severity mistake needs RAG context", 
    },
    "session_id": "session_P001_mid",
    "coaching_history": [],
}

patient_profile_mid = {
    "condition": "knee osteoarthritis",
    "condition_category": "knee",
    "rehab_phase": "mid",
    "pain_level": 4,
    "weeks_into_rehab": 10,
    "age": 58,
    "goals": "Walk dog 30 minutes daily without pain",
}

print("Scenario inputs defined ✓")

Scenario inputs defined ✓


In [8]:
# Build contexts
context_early = merge_to_patient_context(coaching_event_early, patient_profile_early)
context_mid = merge_to_patient_context(coaching_event_mid, patient_profile_mid)

# Generate coaching
print("Running Scenario A (Early Phase)...")
output_early = agent.generate_coaching(context_early)

print("\nRunning Scenario B (Mid Phase)...")
output_mid = agent.generate_coaching(context_mid)

print("\nBoth scenarios complete ✓")

Running Scenario A (Early Phase)...

Generating coaching for patient: session_P001_early
Condition: knee osteoarthritis | Phase: early
Pain: 7/10 | Week: 3

[1/4] RAG query: 'knee osteoarthritis exercises early rehabilitation squat progression'
[2/4] Retrieved 3 source(s): ['Home Exercise Programs for Musculoskeletal and Sports Injuries The Evidence-Based Guide for Practitioners (Ian W. Wender (Editor), James F. Wyss (Editor)) (z-library.sk, 1lib.sk, z-lib.sk).txt', 'Therapeutic Exercise Foundations and Techniques (Carolyn Kisner, Lynn Allen Colby) (z-library.sk, 1lib.sk, z-lib.sk).txt', '9---Lower-Extremity-Workbook-Photograph-of-a-person-stepping-onto-_2019_Ther.txt']
[3/4] Generating coaching (gemma3:4b)...
[4/4] Polishing response...

✓ Done in 17.6s

Running Scenario B (Mid Phase)...

Generating coaching for patient: session_P001_mid
Condition: knee osteoarthritis | Phase: mid
Pain: 4/10 | Week: 10

[1/4] RAG query: 'knee osteoarthritis exercises mid rehabilitation squat progressi

In [ ]:
# ── Final Progress Narrative Report ──────────────────────────────────────
def generate_progress_report(output_early, output_mid, patient_profile):
    """
    Compare two CoachingOutputs from the same patient at different phases.
    Produces a structured progress narrative.
    """

    early_prof = patient_profile_early
    mid_prof = patient_profile_mid

    # Pain improvement
    pain_delta = early_prof["pain_level"] - mid_prof["pain_level"]
    pain_trend = (
        "improved" if pain_delta > 0 else "unchanged" if pain_delta == 0 else "worsened"
    )

    # Week progression
    weeks_delta = mid_prof["weeks_into_rehab"] - early_prof["weeks_into_rehab"]

    # Exercise progression (early: mini squat → mid: full squat)
    ex_early = context_early.recent_exercises[0]
    ex_mid = context_mid.recent_exercises[0]

    # Quality score progression
    q_early = coaching_event_early["coaching_event"]["quality_score"]
    q_mid = coaching_event_mid["coaching_event"]["quality_score"]
    q_delta = q_mid - q_early

    # Severity progression
    sev_early = coaching_event_early["coaching_event"]["severity"]
    sev_mid = coaching_event_mid["coaching_event"]["severity"]

    # ROM progression
    rom_early = coaching_event_early["coaching_event"]["metrics"]["rom_level"]
    rom_mid = coaching_event_mid["coaching_event"]["metrics"]["rom_level"]

    report = {
        "patient_id": output_early.patient_id.split("_")[0] + "_P001",
        "condition": early_prof["condition"],
        "goal": early_prof["goals"],
        "period_weeks": f"Week {early_prof['weeks_into_rehab']} → Week {mid_prof['weeks_into_rehab']}",
        "phase_progression": f"{early_prof['rehab_phase'].upper()} → {mid_prof['rehab_phase'].upper()}",
        "pain": {
            "early": early_prof["pain_level"],
            "mid": mid_prof["pain_level"],
            "delta": -pain_delta,  # negative = improvement
            "trend": pain_trend,
        },
        "exercise_progression": {
            "early": f"{ex_early.name} — {ex_early.difficulty_feedback}",
            "mid": f"{ex_mid.name} — {ex_mid.difficulty_feedback}",
        },
        "form_quality": {
            "early_score": q_early,
            "mid_score": q_mid,
            "delta": round(q_delta, 2),
            "trend": "improving" if q_delta > 0 else "declining",
        },
        "mistake_progression": {
            "early": f"{sev_early} severity — {coaching_event_early['coaching_event']['mistake']['type']}",
            "mid": f"{sev_mid} severity — {coaching_event_mid['coaching_event']['mistake']['type']}",
        },
        "rom_progression": {
            "early": rom_early,
            "mid": rom_mid,
            "trend": "improved" if rom_mid > rom_early else "unchanged",
        },
        "sources_used": list(
            set(output_early.retrieved_sources + output_mid.retrieved_sources)
        ),
        "coaching_snapshots": {
            "early_phase": output_early.coaching_feedback,
            "mid_phase": output_mid.coaching_feedback,
        },
    }
    return report


report = generate_progress_report(output_early, output_mid, patient_profile_early)

# ── Print structured report ───────────────────────────────────────────────
print("=" * 65)
print("  PATIENT PROGRESS REPORT")
print("=" * 65)
print(f"  Patient:     {report['patient_id']}")
print(f"  Condition:   {report['condition']}")
print(f"  Goal:        {report['goal']}")
print(f"  Period:      {report['period_weeks']}  ({report['phase_progression']})")

print(f"\n  📉 Pain Level")
print(f"     Early  : {report['pain']['early']}/10")
print(f"     Mid    : {report['pain']['mid']}/10")
print(
    f"     Change : {'+' if report['pain']['delta'] > 0 else ''}{report['pain']['delta']} pts  ({report['pain']['trend']})"
)

print(f"\n  🏋️  Exercise Progression")
print(f"     Early  : {report['exercise_progression']['early']}")
print(f"     Mid    : {report['exercise_progression']['mid']}")

print(f"\n  📐 Form Quality Score (0–1)")
print(f"     Early  : {report['form_quality']['early_score']:.2f}")
print(f"     Mid    : {report['form_quality']['mid_score']:.2f}")
print(
    f"     Change : {'+' if report['form_quality']['delta'] > 0 else ''}{report['form_quality']['delta']}  ({report['form_quality']['trend']})"
)

print(f"\n  ⚠️  Mistake Progression")
print(f"     Early  : {report['mistake_progression']['early']}")
print(f"     Mid    : {report['mistake_progression']['mid']}")

print(f"\n  🦵 Range of Motion (1–3 scale)")
print(f"     Early  : {report['rom_progression']['early']}")
print(f"     Mid    : {report['rom_progression']['mid']}")
print(f"     Trend  : {report['rom_progression']['trend']}")

print(f"\n  📚 Knowledge Sources Used")
for s in report["sources_used"]:
    print(f"     • {s}")

print(f"\n{'=' * 65}")
print("  COACHING SNAPSHOTS")
print("=" * 65)
print(f"\n  [ EARLY PHASE — Week {patient_profile_early['weeks_into_rehab']} ]")
print(f"  {'-' * 60}")
print(report["coaching_snapshots"]["early_phase"])
print(f"\n  [ MID PHASE — Week {patient_profile_mid['weeks_into_rehab']} ]")
print(f"  {'-' * 60}")
print(report["coaching_snapshots"]["mid_phase"])
print(f"\n{'=' * 65}")

  PATIENT PROGRESS REPORT
  Patient:     session_P001
  Condition:   knee osteoarthritis
  Goal:        Walk dog 30 minutes daily without pain
  Period:      Week 3 → Week 10  (EARLY → MID)

  📉 Pain Level
     Early  : 7/10
     Mid    : 4/10
     Change : -3 pts  (improved)

  🏋️  Exercise Progression
     Early  : Mini Squat — too hard — knee valgus detected
     Mid    : Squat — too hard — forward lean detected

  📐 Form Quality Score (0–1)
     Early  : 0.28
     Mid    : 0.35
     Change : +0.07  (improving)

  ⚠️  Mistake Progression
     Early  : high severity — knee valgus
     Mid    : medium severity — forward lean

  🦵 Range of Motion (1–3 scale)
     Early  : 1
     Mid    : 2
     Trend  : improved

  📚 Knowledge Sources Used
     • Home Exercise Programs for Musculoskeletal and Sports Injuries The Evidence-Based Guide for Practitioners (Ian W. Wender (Editor), James F. Wyss (Editor)) (z-library.sk, 1lib.sk, z-lib.sk).txt
     • Therapeutic Exercise Foundations and Techni

In [ ]:
output_upstream = agent.generate_coaching(context_upstream)
print(output_upstream.coaching_feedback)


Generating coaching for patient: test_session_123
Condition: knee osteoarthritis | Phase: mid
Pain: 4/10 | Week: 10

[1/4] RAG query: 'knee osteoarthritis exercises mid rehabilitation squat progression'
[2/4] Retrieved 2 source(s): ['9---Lower-Extremity-Workbook-Photograph-of-a-person-stepping-onto-_2019_Ther.txt', 'Therapeutic Exercise Foundations and Techniques (Carolyn Kisner, Lynn Allen Colby) (z-library.sk, 1lib.sk, z-lib.sk).txt']
[3/4] Generating coaching (gemma3:4b)...
[4/4] Polishing response...

✓ Done in 17.8s
Okay, let’s see what we can do to get you feeling even better on your walk with your dog! It’s fantastic to hear you’ve been consistently working on your squats – it’s brilliant you’ve reached week 10! I understand your frustration with the forward lean; it’s really common with knee osteoarthritis, especially as we try to increase our range of motion.

**Feedback & Modifications:** Let’s continue focusing on controlled movement during your squats. It’s brilliant you’r

In [ ]:
# context_knee = make_sample_context('knee')
# print('Patient context:')
# print(f'  Condition: {context_knee.condition}')
# print(f'  Phase: {context_knee.rehab_phase.value}')
# print(f'  Message: "{context_knee.patient_message}"')

Patient context:
  Condition: knee osteoarthritis
  Phase: mid
  Message: "I finished most exercises but the mini squats hurt my knee going down. Also I'm struggling with stairs at home."


In [ ]:
# output = agent.generate_coaching(context_knee)

# print('='*60)
# print('COACHING FEEDBACK')
# print('='*60)
# print(output.coaching_feedback)
# print()
# print(f'Sources: {output.retrieved_sources}')
# print(f'Confidence: {output.confidence_score:.0%}')


Generating coaching for patient: P001
Condition: knee osteoarthritis | Phase: mid
Pain: 4/10 | Week: 10

[1/4] RAG query: 'knee osteoarthritis exercises mid rehabilitation stair climbing pain management squat progression'
[2/4] Retrieved 2 source(s): ['9---Lower-Extremity-Workbook-Photograph-of-a-person-stepping-onto-_2019_Ther.txt', 'Therapeutic Exercise Foundations and Techniques (Carolyn Kisner, Lynn Allen Colby) (z-library.sk, 1lib.sk, z-lib.sk).txt']
[3/4] Generating coaching (gemma3:4b)...
[4/4] Polishing response...

✓ Done in 15.2s
COACHING FEEDBACK
Hi [Patient Name],

Thank you so much for sharing such a detailed update on your session – it’s fantastic to hear you’ve been consistently working through most of your exercises and that you’ve been so open about how they felt. It sounds like you’re making really good progress overall!

Let’s keep focusing on those fantastic quad sets and straight leg raises – you’re feeling them well, and that’s brilliant. Because the mini squats 

### Scenario B: Shoulder Rehab (Week 3)

In [6]:
output_shoulder = agent.generate_coaching(make_sample_context('shoulder'))
print(output_shoulder.coaching_feedback)


Generating coaching for patient: P002
Condition: rotator cuff tendinopathy | Phase: early
Pain: 5/10 | Week: 3

[1/4] RAG query: 'rotator cuff tendinopathy exercises early rehabilitation pain management activity modification shoulder exercises'
[2/4] Retrieved 2 source(s): ['Therapeutic Exercise for Musculoskeletal Injuries 4th Edition (Peggy A. Houglum) (z-library.sk, 1lib.sk, z-lib.sk).txt', 'Therapeutic Exercise for the Shoulder - Physiopedia.txt']
[3/4] Generating coaching (gemma3:4b)...
[4/4] Polishing response...

✓ Done in 13.0s
Okay, let’s see how we’re doing! It’s fantastic that you’ve been consistently completing your exercises this week, and I really appreciate you sharing how you’re feeling. It’s completely understandable that you’re experiencing tightness in the morning and that the external rotation is causing a sharp pain at the end range – that’s a really important signal for us to pay attention to.

**Feedback & Modifications:**

Given that the external rotation felt 

### Scenario C: ACL Recovery (Week 20 — Return to Sport)

In [7]:
output_acl = agent.generate_coaching(make_sample_context('acl'))
print(output_acl.coaching_feedback)


Generating coaching for patient: P003
Condition: ACL reconstruction recovery | Phase: late
Pain: 2/10 | Week: 20

[1/4] RAG query: 'ACL reconstruction recovery exercises late rehabilitation return to sport plyometric'
[2/4] Retrieved 3 source(s): ['Therapeutic Exercise for Musculoskeletal Injuries 4th Edition (Peggy A. Houglum) (z-library.sk, 1lib.sk, z-lib.sk).txt', 'The_Comprehensive_Manual_of_Therapeutic_Exercises_..._----_(Pg_246--475).txt', 'Therapeutic Exercise Foundations and Techniques (Carolyn Kisner, Lynn Allen Colby) (z-library.sk, 1lib.sk, z-lib.sk).txt']
[3/4] Generating coaching (gemma3:4b)...
[4/4] Polishing response...

✓ Done in 11.5s
Okay, fantastic to hear you’re feeling stronger! It’s really positive that you’re noticing an improvement at week 20 – that’s a huge step in your recovery. Let’s keep building on this momentum.

**Feedback & Next Steps:**

You’ve been doing great with your single leg squats and box jumps, and continuing those is excellent. We’ve seen you

## 5. Custom Patient Input

Build your own PatientContext for testing:

In [8]:
custom_context = PatientContext(
    patient_id='P999',
    condition='patellofemoral pain syndrome',
    condition_category=ConditionCategory.KNEE,
    rehab_phase=RehabPhase.EARLY,
    pain_level=6,
    weeks_into_rehab=4,
    recent_exercises=[
        ExerciseRecord('Clamshells', sets=3, reps=15, completed=True, difficulty_feedback='ok'),
        ExerciseRecord('Wall sits', sets=2, reps=None, completed=False, difficulty_feedback='too hard'),
    ],
    patient_message='The wall sits really hurt behind my kneecap. Should I stop doing them?',
    age=32,
    goals='Run a 5K in 3 months',
)

custom_output = agent.generate_coaching(custom_context)
print(custom_output.coaching_feedback)


Generating coaching for patient: P999
Condition: patellofemoral pain syndrome | Phase: early
Pain: 6/10 | Week: 4

[1/4] RAG query: 'patellofemoral pain syndrome exercises early rehabilitation pain management'
[2/4] Retrieved 1 source(s): ['Therapeutic Exercise for Musculoskeletal Injuries 4th Edition (Peggy A. Houglum) (z-library.sk, 1lib.sk, z-lib.sk).txt']
[3/4] Generating coaching (gemma3:4b)...
[4/4] Polishing response...

✓ Done in 12.5s
Okay, let’s take a look at how week 4 is going! It’s fantastic that you’re consistently completing your clamshells – that’s brilliant work! I understand that the wall sits were too painful behind your kneecap, and it’s completely sensible to stop if they’re causing significant discomfort. Let’s focus on protecting that joint.

**Feedback & Modifications:**

We’re still in the early phase of rehab for patellofemoral pain, and that means prioritizing gentle strengthening and mobility. We’ll continue with the clamshells – 3 sets of 15 reps – and fo

## 6. Inspect Structured Output

In [9]:
# Look at the parsed structured fields
print('📋 Suggested exercises:')
for ex in custom_output.suggested_exercises:
    print(f'  • {ex}')

print('\n⚠️ Safety notes:')
for note in custom_output.safety_notes:
    print(f'  {note}')

print(f'\n💪 Closing motivation:')
print(f'  {custom_output.motivational_note}')

print(f'\n📚 Retrieved from: {custom_output.retrieved_sources}')
print(f'🎯 Confidence score: {custom_output.confidence_score:.0%}')

📋 Suggested exercises:
  • **Glute Bridges:** 3 sets of 10 reps – helps strengthen your glutes, which are crucial for supporting the kneecap.
  • **Hamstring Curls (with resistance band):** 3 sets of 10 reps – strengthens the muscles that help control knee movement.
  • **Heel Slides:** 3 sets of 10 reps – gently improves your range of motion.
  • **Short Arc Quad Contractions:** 3 sets of 10 reps – a key exercise for activating the quadriceps without putting excessive stress on the joint.

⚠️ Safety notes:
  Okay, let’s take a look at how week 4 is going! It’s fantastic that you’re consistently completing your clamshells – that’s brilliant work! I understand that the wall sits were too painful behind your kneecap, and it’s completely sensible to stop if they’re causing significant discomfort. Let’s focus on protecting that joint.
  ⚠️ Note: Remember to always warm up before starting your exercises and to stop immediately if you experience any sharp or worsening pain.

💪 Closing motiva

## 7. Memory-Safe Tips

If you see **kernel crashes** (like in the original trial.ipynb):

1. **Disable polish** — set `enable_polish=False` in CoachingAgent
2. **Reduce retrieval** — set `retrieval_k=2`
3. **Run one scenario at a time** — restart kernel between heavy runs
4. **Check Ollama memory** — `ollama ps` to see VRAM usage
5. **Use `num_predict=300`** — edit coaching_agent.py line ~45

---
## 8. Retrieval & Response Quality Evaluation

Three no-ground-truth metrics evaluated across all 3 demo scenarios:

| Metric | What it measures | Method |
|--------|-----------------|--------|
| **Retrieval Precision / Recall / F1** | Are retrieved chunks relevant to the patient's condition? | Keyword overlap between query terms and chunk content |
| **Source Coverage** | Do returned sources cover the expected document types for this condition? | Expected source keywords vs actual filenames |
| **Response Quality Score** | Is the coaching output structurally complete and safe? | Rule-based checklist (5 criteria) |

In [30]:
# ── Evaluation helpers ────────────────────────────────────────────────────
import re
from typing import List, Dict


# ── 1. Retrieval Precision / Recall / F1 ─────────────────────────────────
#
# FIX — Recall was artificially low because we used k=10 as the denominator,
# but with 1486 chunks in the DB there are far more relevant docs than 10.
#
# New approach: scan ALL docs from the relevant SOURCE FILES for this condition
# (using ChromaDB metadata filter on 'file' field) to get a true denominator.
# This is the standard 'pooling' recall estimation used in IR evaluation.

def build_relevance_keywords(context) -> List[str]:
    """Core condition keywords — kept tight (≤5) so ≥1 match threshold is meaningful."""
    # Use only the most discriminative terms from the condition name
    # keywords = context.condition.lower().split()
    # keywords.append(context.condition_category.value)  # e.g. 'knee', 'shoulder'
    keywords = context.condition.lower().split()   # ['knee', 'osteoarthritis']
    keywords.append(context.condition_category.value)  # 'knee'

    # Add the single most important word from the patient message
    msg_words = re.findall(r'\b[a-z]{5,}\b', context.patient_message.lower())
    stopwords = {'should', 'would', 'could', 'still', 'really', 'going', 'doing'}
    useful = [w for w in msg_words if w not in stopwords]
    keywords.extend(useful[:2])
    return list(set(keywords))


# Maps condition_category → substrings that appear in the ACTUAL filenames
# (derived from trial.ipynb file list)
CONDITION_SOURCE_FILES = {
    "knee": ["houglum", "kisner", "ellenbecker", "home exercise", "lower-extremity"],
    "hip": ["houglum", "kisner", "ellenbecker", "home exercise"],
    "shoulder": ["houglum", "kisner", "physiopedia", "shoulder"],
    "lower_back": ["houglum", "kisner", "home exercise"],
    "mid_back": ["kisner", "houglum"],
    "general_msk": ["kisner", "houglum", "ellenbecker"],
}


def _is_relevant_file(filename: str, category: str) -> bool:
    """Return True if filename matches any known source pattern for this category."""
    fn_lower = filename.lower()
    patterns = CONDITION_SOURCE_FILES.get(category, [])
    return any(p in fn_lower for p in patterns)


def compute_retrieval_metrics(context, output, kb, k: int = 3) -> Dict:
    """Compute Precision, Recall, F1 using file-pool denominator."""
    from prompts import build_rag_query

    query    = build_rag_query(context)
    keywords = build_relevance_keywords(context)
    category = context.condition_category.value

    # -- Retrieved docs (k=3)
    docs = kb.vectordb.similarity_search(query, k=k)

    def chunk_is_relevant(doc) -> bool:
        """A chunk is relevant if its SOURCE FILE is in the condition pool."""
        # fname = doc.metadata.get('file', '')
        # return _is_relevant_file(fname, category)
        text = doc.page_content.lower()
        return sum(1 for kw in keywords if kw in text) >= 1  # 改成 ≥1

    relevant_retrieved = sum(1 for d in docs if chunk_is_relevant(d))
    num_retrieved      = len(docs)

    # -- Total relevant: scan a LARGE pool (k=20) filtered by source file
    # This gives a stable denominator without scanning all 1486 chunks
    pool_docs     = kb.vectordb.similarity_search(query, k=20)
    total_relevant = sum(1 for d in pool_docs if chunk_is_relevant(d))
    total_relevant = max(total_relevant, relevant_retrieved, 1)  # floor at retrieved

    precision = relevant_retrieved / num_retrieved if num_retrieved else 0
    recall    = relevant_retrieved / total_relevant
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0)

    # ── new: P@k for k in [3, 5, 10] ──────────────────────────
    precision_at_k = {}
    for eval_k in [3, 5, 10]:
        k_docs = kb.vectordb.similarity_search(query, k=eval_k)
        k_relevant = sum(1 for d in k_docs if chunk_is_relevant(d))
        precision_at_k[f"P@{eval_k}"] = round(k_relevant / eval_k, 3)
    # ────────────────────────────────────────────────────────────

    return {
        "num_retrieved": num_retrieved,
        "relevant_retrieved": relevant_retrieved,
        "total_relevant_pool": total_relevant,
        "precision": round(precision, 3),
        "recall": round(recall, 3),
        "f1": round(f1, 3),
        "precision_at_k": precision_at_k,  # new
    }


# ── 2. Source Coverage ────────────────────────────────────────────────────
#
# FIX — Original keywords were too short/generic and didn't match actual
# long filenames like 'Osteoarthritis of the Knee (MSK Patient Portal)...'.
#
# New approach: define DOCUMENT GROUPS by substrings of the REAL filenames,
# and check whether AT LEAST ONE group is hit by the returned sources.
# Coverage = (groups hit) / (groups expected for this condition).

# Each entry: condition_category → list of groups.
# A 'group' is a list of filename substrings — any match = group hit.
EXPECTED_SOURCE_GROUPS = {
    "knee": [
        ["houglum", "musculoskeletal injuries"],  # 临床教材
        ["kisner", "foundations"],  # 经典运动治疗
    ],
    "hip": [
        ["houglum", "musculoskeletal injuries"],
        ["kisner", "foundations"],
    ],
    "shoulder": [
        ["houglum", "musculoskeletal injuries"],
        [
            "physiopedia",
            "shoulder",
        ],  # 有 Therapeutic Exercise for the Shoulder - Physiopedia.txt
    ],
    "lower_back": [
        ["kisner", "foundations"],
        ["houglum", "musculoskeletal injuries"],
    ],
    "mid_back": [
        ["kisner", "foundations"],
    ],
    "general_msk": [
        ["kisner", "foundations"],
        ["houglum", "musculoskeletal"],
    ],
}


def compute_source_coverage(context, output) -> Dict:
    """Check whether returned sources cover expected document groups."""
    category       = context.condition_category.value
    expected_groups = EXPECTED_SOURCE_GROUPS.get(category, [])

    if not expected_groups:
        return {'coverage': 1.0, 'matched_groups': [], 'missing_groups': [],
                'sources': output.retrieved_sources}

    sources_lower = ' '.join(s.lower() for s in output.retrieved_sources)
    matched, missing = [], []

    for group in expected_groups:
        hit = any(kw in sources_lower for kw in group)
        if hit:
            matched.append(group)
        else:
            missing.append(group)

    coverage = len(matched) / len(expected_groups)
    return {
        'coverage':       round(coverage, 3),
        'matched_groups': matched,
        'missing_groups': missing,
        'sources':        output.retrieved_sources,
    }


# ── 3. Response Quality Score ─────────────────────────────────────────────
# Checklist of 5 criteria that a good coaching response should satisfy.
# Each criterion is worth 20 points → max score = 100.

QUALITY_CRITERIA = [
    {
        'name': 'Has exercise recommendation',
        'check': lambda text: bool(re.search(
            r'\b(sets?|reps?|exercise|stretch|squat|raise|rotation|walk|bend)\b',
            text, re.IGNORECASE)),
        'weight': 20,
    },
    {
        'name': 'Addresses patient concern',
        'check': lambda text: len(text) > 150,   # proxy: substantive response
        'weight': 20,
    },
    {
        'name': 'Contains safety / caution language',
        'check': lambda text: bool(re.search(
            r'\b(stop|avoid|caution|consult|pain|careful|physiotherapist|doctor)\b',
            text, re.IGNORECASE)),
        'weight': 20,
    },
    {
        'name': 'Has motivational / encouraging tone',
        'check': lambda text: bool(re.search(
            r'\b(great|well done|progress|keep|goal|believe|proud|strong|improve)\b',
            text, re.IGNORECASE)),
        'weight': 20,
    },
    {
        'name': 'Appropriate length (150–500 words)',
        'check': lambda text: 150 <= len(text.split()) <= 500,
        'weight': 20,
    },
]

def compute_response_quality(output) -> Dict:
    """Score the coaching response against the quality checklist."""
    text = output.coaching_feedback
    results = []
    total_score = 0

    for criterion in QUALITY_CRITERIA:
        passed = criterion['check'](text)
        score  = criterion['weight'] if passed else 0
        total_score += score
        results.append({
            'criterion': criterion['name'],
            'passed': passed,
            'score': score,
        })

    word_count = len(text.split())
    return {
        'total_score':  total_score,          # 0–100
        'max_score':    100,
        'percentage':   total_score / 100,
        'word_count':   word_count,
        'criteria':     results,
    }

print('Evaluation functions defined ✓')

Evaluation functions defined ✓


In [31]:
# ── Run evaluation across all 3 scenarios ────────────────────────────────
# Assumes output, output_shoulder, output_acl are already generated above.

scenarios = [
    ('Knee OA',   make_sample_context('knee'),     output),
    ('Shoulder',  make_sample_context('shoulder'), output_shoulder),
    ('ACL',       make_sample_context('acl'),      output_acl),
]

all_results = []

for label, ctx, out in scenarios:
    ret  = compute_retrieval_metrics(ctx, out, kb, k=3)
    cov  = compute_source_coverage(ctx, out)
    qual = compute_response_quality(out)
    all_results.append({
        'scenario': label,
        'retrieval': ret,
        'coverage':  cov,
        'quality':   qual,
    })

print(f'Evaluated {len(all_results)} scenarios ✓')

Evaluated 3 scenarios ✓


In [32]:
# ── Print full report ─────────────────────────────────────────────────────

def bar(value, width=30):
    """Text progress bar for a 0–1 value."""
    filled = int(value * width)
    return '█' * filled + '░' * (width - filled)


for r in all_results:
    s   = r['scenario']
    ret = r['retrieval']
    cov = r['coverage']
    q   = r['quality']

    print(f'\n{"="*65}')
    print(f'  SCENARIO: {s}')
    print(f'{"="*65}')

    # — Retrieval metrics
    print(f'\n  📡 RETRIEVAL  (k=3, proxy-label method)')
    print(f'     Retrieved: {ret["num_retrieved"]}  |  '
          f'Relevant: {ret["relevant_retrieved"]}  |  '
          f'Pool relevant (k=20): {ret["total_relevant_pool"]}')
    print(f'     Precision  {bar(ret["precision"])}  {ret["precision"]:.1%}')
    print(f'     Recall     {bar(ret["recall"])}  {ret["recall"]:.1%}')
    print(f'     F1         {bar(ret["f1"])}  {ret["f1"]:.1%}')

    # new
    print(f"\n     Precision @ k breakdown:")
    for label, val in ret["precision_at_k"].items():
        print(f"       {label}  {bar(val)}  {val:.1%}")
    
    # — Source coverage
    print(f'\n  📚 SOURCE COVERAGE')
    print(f'     Sources returned: {cov["sources"]}')
    print(f'     Coverage  {bar(cov["coverage"])}  {cov["coverage"]:.1%}')
    if cov['missing_groups']:
        print(f'     ⚠️  Missing doc types: {cov["missing_groups"]}')
    else:
        print(f'     ✓  All expected source types covered')

    # — Response quality
    print(f'\n  ✍️  RESPONSE QUALITY  (word count: {q["word_count"]})')
    print(f'     Score     {bar(q["percentage"])}  {q["total_score"]}/100')
    for c in q['criteria']:
        icon = '✓' if c['passed'] else '✗'
        print(f'     {icon}  {c["criterion"]:45s} +{c["score"]}')

print(f'\n{"="*65}')


  SCENARIO: Knee OA

  📡 RETRIEVAL  (k=3, proxy-label method)
     Retrieved: 3  |  Relevant: 3  |  Pool relevant (k=20): 17
     Precision  ██████████████████████████████  100.0%
     Recall     █████░░░░░░░░░░░░░░░░░░░░░░░░░  17.6%
     F1         █████████░░░░░░░░░░░░░░░░░░░░░  30.0%

     Precision @ k breakdown:
       P@3  ██████████████████████████████  100.0%
       P@5  ████████████████████████░░░░░░  80.0%
       P@10  ███████████████████████████░░░  90.0%

  📚 SOURCE COVERAGE
     Sources returned: ['9---Lower-Extremity-Workbook-Photograph-of-a-person-stepping-onto-_2019_Ther.txt', 'Therapeutic Exercise Foundations and Techniques (Carolyn Kisner, Lynn Allen Colby) (z-library.sk, 1lib.sk, z-lib.sk).txt']
     Coverage  ███████████████░░░░░░░░░░░░░░░  50.0%
     ⚠️  Missing doc types: [['houglum', 'musculoskeletal injuries']]

  ✍️  RESPONSE QUALITY  (word count: 268)
     Score     ██████████████████████████████  100/100
     ✓  Has exercise recommendation                   

In [33]:
# ── Summary table across all scenarios ───────────────────────────────────
import statistics

print(f'\n{"="*65}')
print('  SUMMARY TABLE')
print(f'{"="*65}')
print(f'  {"Scenario":<12} {"Precision":>10} {"Recall":>8} {"F1":>8} {"Coverage":>10} {"Quality":>9}')
print(f'  {"-"*62}')

for r in all_results:
    ret = r['retrieval']
    cov = r['coverage']
    q   = r['quality']
    print(f'  {r["scenario"]:<12} '
          f'{ret["precision"]:>9.1%} '
          f'{ret["recall"]:>8.1%} '
          f'{ret["f1"]:>8.1%} '
          f'{cov["coverage"]:>9.1%} '
          f'{q["total_score"]:>7}/100')

# Averages
avg_p   = statistics.mean(r['retrieval']['precision'] for r in all_results)
avg_r   = statistics.mean(r['retrieval']['recall']    for r in all_results)
avg_f1  = statistics.mean(r['retrieval']['f1']        for r in all_results)
avg_cov = statistics.mean(r['coverage']['coverage']   for r in all_results)
avg_q   = statistics.mean(r['quality']['total_score'] for r in all_results)

print(f'  {"-"*62}')
print(f'  {"AVERAGE":<12} {avg_p:>9.1%} {avg_r:>8.1%} {avg_f1:>8.1%} {avg_cov:>9.1%} {avg_q:>7.1f}/100')
print(f'{"="*65}')

# Overall grade
overall = (avg_f1 * 0.35 + avg_cov * 0.25 + avg_q / 100 * 0.40)
grade = 'A' if overall >= 0.80 else 'B' if overall >= 0.65 else 'C' if overall >= 0.50 else 'D'
# Note: F1 is weighted more heavily than coverage and quality because it's the most direct measure of retrieval effectiveness, which is the foundation for good coaching. Coverage and quality are also important but somewhat downstream of retrieval performance.
print(f"\n  📈 Average Precision @ k:")
for pk_label in ["P@3", "P@5", "P@10"]:
    avg_pk = statistics.mean(
        r["retrieval"]["precision_at_k"][pk_label] for r in all_results
    )
    print(f"     {pk_label}  {bar(avg_pk)}  {avg_pk:.1%}")

print(f'\n  🎯 Overall Weighted Score: {overall:.1%}  →  Grade: {grade}')
print(f'     (F1×0.35 + Coverage×0.25 + Quality×0.40)\n')


  SUMMARY TABLE
  Scenario      Precision   Recall       F1   Coverage   Quality
  --------------------------------------------------------------
  Knee OA         100.0%    17.6%    30.0%     50.0%     100/100
  Shoulder        100.0%    15.0%    26.1%    100.0%     100/100
  ACL             100.0%    15.0%    26.1%    100.0%     100/100
  --------------------------------------------------------------
  AVERAGE         100.0%    15.9%    27.4%     83.3%   100.0/100

  📈 Average Precision @ k:
     P@3  ██████████████████████████████  100.0%
     P@5  ████████████████████████████░░  93.3%
     P@10  █████████████████████████████░  96.7%

  🎯 Overall Weighted Score: 70.4%  →  Grade: B
     (F1×0.35 + Coverage×0.25 + Quality×0.40)

